# 06 · Use: search over sanitized text

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/06_search_on_sanitized_data.ipynb)

**Runtime:** CPU is fine. It runs one detector (Presidio by default) over a small corpus.

← [05_sanitization](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/05_sanitization.ipynb) · [series index](https://github.com/jstjoe/local-privacy/blob/main/notebooks/README.md)

Downstream tools such as search, retrieval, RAG and joins run on the
*sanitized* text. This notebook builds a small BM25 search engine over a
~130-document corpus, sanitizes the documents and the queries under each mode,
and measures how many of the right documents come back.

The corpus is designed to make the difference visible. A few "gold" documents
mention the people the queries ask about. The filler documents are full of
*other* emails, phone numbers and addresses. So a mode that can't tell PII
values apart can't find the gold documents.

In [ ]:
# Setup — run once per session. On Colab this clones and installs the harness
# (a few minutes the first time). Locally, after `uv sync`, it does nothing.
HARNESS_REPO = "https://github.com/jstjoe/local-privacy.git"
HARNESS_REF = "main"   # branch or tag — e.g. a PR branch to try it before merging
OPF_REPO = "https://github.com/openai/privacy-filter.git"
OPF_REF = "main"
USE_DRIVE = False      # True: keep results in Google Drive so the other notebooks reuse them

import importlib.util, os, subprocess, sys

if importlib.util.find_spec("opf_eval") is None:
    home = "/content" if "google.colab" in sys.modules else os.path.expanduser("~")
    for repo, ref, dest in [(OPF_REPO, OPF_REF, f"{home}/privacy-filter"),
                            (HARNESS_REPO, HARNESS_REF, f"{home}/local-privacy")]:
        if not os.path.exists(dest):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", ref, repo, dest], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{home}/privacy-filter",
                    f"{home}/local-privacy/eval[notebooks]", f"{home}/local-privacy/api"], check=True)
    # A running kernel doesn't always see freshly pip-installed packages;
    # putting the sources on sys.path makes the imports below work either way.
    sys.path[:0] = [f"{home}/privacy-filter", f"{home}/local-privacy/eval/src", f"{home}/local-privacy/api/src"]

from opf_eval import nb
ws = nb.setup(drive=USE_DRIVE)

## Corpus and queries

Each query has a gold relevance set: the document indices that should come
back.

In [ ]:
from opf_eval.demo import search

corpus, queries = search.build_corpus(seed=42)
TOP_K = 5
print(f"{len(corpus)} documents, {len(queries)} queries, top-{TOP_K}")
for q in queries:
    print(f"  {q.text!r}: relevant docs {sorted(q.relevant)}")
print("\nsample filler:", corpus[10])

## Sanitize both sides

Documents and queries go through the **same** detector and the same renderer,
and for `label_token` the same vault. Otherwise the tokens wouldn't line up. A
`plain` row keeps the unsanitized text as a no-privacy baseline.

In [ ]:
from opf_eval.detectors import registry

S = nb.load_session(quiet=True)
TOKEN_VAULT = nb.token_vault()

# === EDIT ME ===
DETECTOR = "presidio" if "presidio" in S.detectors else S.detectors[0]
# ================
if DETECTOR.startswith("presidio"):
    nb.ensure_spacy_model("en_core_web_lg")
det = registry.build(DETECTOR, device=S.device or nb.device())

MODES = ["redact", "label", "label_number"] + (["label_token"] if TOKEN_VAULT else [])
if not TOKEN_VAULT:
    print("label_token skipped — token vault not configured (see 05)")

docs_by_mode = search.sanitize_all(det, corpus, MODES, TOKEN_VAULT)
queries_by_mode = search.sanitize_all(det, [q.text for q in queries], MODES, TOKEN_VAULT)

for mode in docs_by_mode:
    print(f"{mode:>13}  doc 0: {docs_by_mode[mode][0]}")
    print(f"{'':>13}  query: {queries_by_mode[mode][0]}")

## Index and search

There's one BM25 index per mode, and each query runs against the index built
under its own mode. The tokenizer keeps bracketed tokens like `[EMAIL_u8UBDWQ]`
whole, so a vault token in a query matches the same token in a document.

In [ ]:
retrievals = search.retrieve(docs_by_mode, queries_by_mode, k=TOP_K)
rows = search.score(queries, queries_by_mode, retrievals)

for qi, q in enumerate(queries):
    table = [
        {
            "mode": r["mode"],
            "sanitized query": r["sanitized_query"],
            f"top-{TOP_K}": ", ".join(("✓" if i in q.relevant else "✗") + str(i) for i in r["retrieved"]),
            "precision": r["precision"],
            "recall": r["recall"],
        }
        for r in rows if r["query"] == qi
    ]
    nb.show_md(f"### Query {qi + 1}: `{q.text}` — relevant {sorted(q.relevant)}")
    nb.show_table(table, floatfmt=".2f")

## Verdict

This is the average over both queries, computed from the run above rather than
hard-coded, so it reflects the detector you picked.

In [ ]:
nb.show_table(search.verdict(rows), floatfmt=".2f")

**Why the modes behave the way they do**

- **`redact`:** every query becomes `********`, whichever email it asked
  about, so the alice and charlie queries return the same documents.
  `********` also appears in almost every document, which gives it a negative
  IDF. Short gold documents are then penalised *more* by length normalisation
  than long filler documents, and fall out of the top-k.
- **`label`:** a query becomes `[EMAIL]`. That separates "an email" from "a
  phone number", but not alice from charlie. Filler documents with several
  emails have a higher term frequency and outrank the gold ones.
- **`label_number`:** numbering restarts in each document. If alice is the
  second email in a document she's `[EMAIL_2]`, and the query's `[EMAIL_1]`
  misses her. It also matches every document whose *first* email is someone
  else. Hits happen by coincidence of position, not identity.
- **`label_token`:** each value gets one deterministic token across every
  document and query. The alice query matches the documents that mention
  alice and nothing else. This property is **referential integrity**, and
  it's what search, joins and RAG over sanitized data depend on.

If detection misses the email in a query or document, every mode, including
`label_token`, falls back to the plaintext for that span. Sanitized search is
only as good as the detector underneath.

## Where next

- **Your own corpus:** swap `corpus` and `queries` for your documents and real
  queries, with hand-marked relevant ids, and re-run this notebook.
- **Your own data through the whole series:** register it in
  [01](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/01_datasets.ipynb), benchmark it in 02–03, and pick a detector or
  composite that fits it rather than a public benchmark.
- **RAG:** add a generation step. Retrieve sanitized chunks, answer with an
  LLM, and score the answers. The open question is whether models reason as
  well over `[EMAIL_u8UBDWQ]`-style tokens as over plaintext.